# 🧬 ProteinMPNN Tool Example

This notebook demonstrates how to use **ProteinMPNN** for protein sequence design (inverse folding).

## 📖 What is ProteinMPNN?

[ProteinMPNN](https://www.science.org/doi/10.1126/science.add2187) is a state-of-the-art deep learning model for designing protein sequences that fold into desired 3D structures.

### Key Features:
- **Inverse folding** -- Design sequences that fold into target structures
- **High success rate** -- Experimentally validated designs with high expression and stability
- **Sampling** -- Generate multiple diverse sequence candidates
- **Flexible control** -- Temperature, batch size, and chain-specific design options

## 📥 Imports

## 📦 Shared Data Types

### `Structure`
A protein structure loaded from a PDB/CIF file path or PDB content string. Automatically resolved on construction.

### `InverseFoldingStructureInput`
Bundles a structure with its per-structure design constraints.

| Field | Type | Description |
|-------|------|-------------|
| `structure` | `Structure` | Protein structure. Accepts a file path, PDB content string, or `Structure` object |
| `chain_ids` | `Optional[List[str]]` | Chain IDs to design. If `None`, all chains are designed |
| `fixed_positions` | `Optional[Dict[str, List[int]]]` | Chain ID to residue positions (1-indexed) to keep fixed during design |

In [7]:
from proto_tools import (
    InverseFoldingConfig,
    InverseFoldingInput,
    InverseFoldingStructureInput,
    run_proteinmpnn_sample,
)
from proto_tools.entities.structures.examples import get_gfp_structure

## 🔧 1. Prepare Input

Define the target structure and sampling parameters.

### API Reference

**`InverseFoldingInput`**

| Field | Type | Description |
|-------|------|-------------|
| `inputs` | `List[InverseFoldingStructureInput]` | List of inverse folding inputs, each containing a structure and constraints |

**`InverseFoldingConfig`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `num_sequences_per_structure` | `int` | `1` | Total number of sequences to generate per structure |
| `batch_size` | `Optional[int]` | `None` | Max sequences per GPU forward pass (defaults to `num_sequences_per_structure`) |
| `temperature` | `float` | `0.1` | Controls randomness in sampling (0.0 to 1.0) |
| `excluded_amino_acids` | `Optional[List[str]]` | `None` | Amino acids to exclude from designed sequences |

**`InverseFoldingOutput`**

| Field | Type | Description |
|-------|------|-------------|
| `designed_sequences` | `List[DesignedSequences]` | Designed sequence results, one per input structure |

**`ProteinMPNNSequences`** (extends `DesignedSequences`)

| Field | Type | Description |
|-------|------|-------------|
| `sequences` | `List[str]` | Designed amino acid sequences |
| `perplexity` | `List[float]` | Per-sequence perplexity values |
| `sequence_identity` | `List[float]` | Sequence identity to the PDB prompt sequence |

In [ ]:
# Use the GFP (Green Fluorescent Protein) structure as input
gfp_structure = get_gfp_structure()
struct_input = InverseFoldingStructureInput(structure=gfp_structure, chain_ids=["A"])
inputs = InverseFoldingInput(inputs=[struct_input])

# Configure sampling parameters
config = InverseFoldingConfig(
    num_sequences_per_structure=2,  # Generate 2 sequence designs
    temperature=0.1                 # Conservative designs
)

## 🚀 2. Run ProteinMPNN Sampling

Generate new sequences that fit the backbone structure.

ProteinMPNN analyzes the 3D structure and designs sequences optimized to fold into that shape. Each design is a new amino acid sequence that should adopt the same fold as the input structure.

In [9]:
# Visualize the GFP backbone structure used for sequence design
gfp_structure.visualize(style="cartoon", color_by="chain")

In [10]:
result = run_proteinmpnn_sample(inputs, config)

# Display the designed sequences
for i, seq_res in enumerate(result.designed_sequences):
    print(f"\nStructure {i} designs:")
    for j, seq in enumerate(seq_res.sequences, 1):
        print(f"  Design {j}: {seq[:50]}..." if len(seq) > 50 else f"  Design {j}: {seq}")
        print(f"            Length: {len(seq)} residues")


Structure 0 designs:
  Design 1: SEGEELFKGVVPIKVRLDGDVNGKKFSIEGEGEGLATEGKLRLKFVCTTG...
            Length: 227 residues
  Design 2: SKGAELFKGVVPIEVDLDGDVNGHKFRVKGEGEGDATEGKIRLKFVCTTG...
            Length: 227 residues


In [11]:
# Compare designed sequence with the original sequence in the structure
chain_a = gfp_structure.get_chain_sequence("A", remove_non_standard=True) # remove non-standard removes HETATMs

for ds_ind, designed_sequence in enumerate(result.designed_sequences[0]):
    print(f"Design {ds_ind + 1}:")
    print(f"Designed: {designed_sequence}")
    
    # Create alignment line
    alignment = ""
    for i, (d_char, o_char) in enumerate(zip(designed_sequence, chain_a)):
        if d_char == o_char:
            alignment += "|"
        else:
            alignment += "-"
    
    # Pad alignment if sequences have different lengths
    if len(designed_sequence) < len(chain_a):
        alignment += "-" * (len(chain_a) - len(designed_sequence))
    
    print(f"          {alignment}")
    print(f"Original: {chain_a}")
    print()

Design 1:
Designed: SEGEELFKGVVPIKVRLDGDVNGKKFSIEGEGEGLATEGKLRLKFVCTTGKLPVPWPTLVPFLIQCFTRYPEHMKQHDFFKSAMPEGYVREATAYFENDGNFKSRAEVYMDGDTLVNEIELKGSDFKEDGNILGHKLKYSYGSSKRYITADPAKNGIKVEYTLEYPVEDGSVQKVKYEEQNTPIGDGPVLLPEPHYIETSSKLSKDPNEKRDHMVLDETMVAAGIPE
          |-|||||-|||||-|-|||||||-|||--|||||-||-|||-|||-||||||||||||||--|----------------|---------------|--|--------------------------------------------------------------------------------|----------|-----------------------------|-|-----
Original: SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLtVQCFARYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNHHKVYITADKQKNGIKVNFKTRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLHTHSKLSKDPNEKRDHMVLLEFVTAAGITL

Design 2:
Designed: SKGAELFKGVVPIEVDLDGDVNGHKFRVKGEGEGDATEGKIRLKFVCTTGKLPVPWPTLITTLIQCFTRYPEHMKQHDFFKSAMPEGYVREQTLNFKDDGNYKTRAEVKFEGDTLVNRIELKGTDFKPDGNILGHKLEYSAGSSKVNITADPKNNGIKENFTLEYKVKDGSTQKVDVKATNTPIGDGPVLLPEPHYIETSTKLSKDPNEKRDHMVLDETQVAAGIPE
          |||-|||-|||||-|

## 💾 3. Export Results

Save the designed sequences to a file for downstream analysis.

### Supported formats:
- **JSON** -- Structured data with metadata and all design information
- **FASTA** -- Simple sequence format for compatibility with other tools

The exported sequences can be used for:
- Experimental validation (gene synthesis and expression)
- Further computational analysis (structure prediction, stability assessment)
- Comparison with native sequences

In [ ]:
# Export to JSON format
result.export("proteinmpnn_designs", export_path="./example_output", file_format="json")
print("Results exported to ./example_output/proteinmpnn_designs/")